In [5]:
import json
from tabulate import tabulate

# GQS Analysis on 24 Hour Logs
EDA on the logs created from separate GQS runs.
## Neo4j 5.20

In [6]:
with open('src/gqs_24hour_run.log.txt') as file:
    lines = [line.rstrip() for line in file]

len(lines)

14583

In [7]:
nodes_created = [l for l in lines if l.startswith('CREATE (')]
constraints_created = [l for l in lines if l.startswith('CREATE CONSTRAINT')]
indexes_created = [l for l in lines if l.startswith('CREATE INDEX')]
text_indexes_created = [l for l in lines if l.startswith('CREATE TEXT INDEX')]
matches_made = [l for l in lines if l.startswith('MATCH')]

In [8]:
print(f"Nodes created: {len(nodes_created)}")
print(f"Constraints created: {len(constraints_created)}")
print(f"Indexes created: {len(indexes_created)}")
print(f"Text indexes created: {len(text_indexes_created)}")
print(f"Matches made: {len(matches_made)}")

Nodes created: 416
Constraints created: 1188
Indexes created: 2518
Text indexes created: 751
Matches made: 5286


## Neo4j 4.4

In [9]:
with open('src/gqs_v4_24hour_run.log.txt') as file:
    lines_4 = [line.rstrip() for line in file]

len(lines_4)

11368

In [10]:
nodes_created_4 = [l for l in lines_4 if l.startswith('CREATE (')]
constraints_created_4 = [l for l in lines_4 if l.startswith('CREATE CONSTRAINT')]
indexes_created_4 = [l for l in lines_4 if l.startswith('CREATE INDEX')]
text_indexes_created_4 = [l for l in lines_4 if l.startswith('CREATE TEXT INDEX')]
matches_made_4 = [l for l in lines_4 if l.startswith('MATCH')]

In [11]:
print(f"Nodes created: {len(nodes_created_4)}")
print(f"Constraints created: {len(constraints_created_4)}")
print(f"Indexes created: {len(indexes_created_4)}")
print(f"Text indexes created: {len(text_indexes_created_4)}")
print(f"Matches made: {len(matches_made_4)}")

Nodes created: 381
Constraints created: 1221
Indexes created: 2565
Text indexes created: 767
Matches made: 3782


# 24 Hour Log Comparison
EDA on created comparion log from `compare_24hr_logs.py`.

In [12]:
with open('src/gqs_24hr_comparison.json', 'r') as file:
    log = json.load(file)

neo4j_4_4 = log['v4']
neo4j_5_20 = log['v5']
[i for i in neo4j_5_20]  # see what each run logged

['version',
 'match_queries',
 'create_statements',
 'schema_statements',
 'result_sizes',
 'nonempty_count',
 'nonempty_rate',
 'bug_reports',
 'global_state_resets']

# GQS Comparison
Comparing the characteristics from the logs.

In [13]:
print(tabulate([['Nodes', len(nodes_created_4), len(nodes_created)], 
                ['Constraints', len(constraints_created_4), len(constraints_created)], 
                ['Indexes', len(indexes_created_4), len(indexes_created)], 
                ['Text indexes', len(text_indexes_created_4), len(text_indexes_created)], 
                ['Matches', len(matches_made_4), len(matches_made)], 
                ['Non-empty count', neo4j_4_4['nonempty_count'], neo4j_5_20['nonempty_count']], 
                ['Non-empty rate', neo4j_4_4['nonempty_rate'], neo4j_5_20['nonempty_rate']], 
                ['Bug reports', neo4j_4_4['bug_reports']/2, neo4j_5_20['bug_reports']/2],  # traceback message also shows bug name 
                ['Global state resets', neo4j_4_4['global_state_resets'], neo4j_5_20['global_state_resets']]], 
               headers=['Created', 'Neo4j 4.4', 'Neo4j 5.20']))

Created                Neo4j 4.4    Neo4j 5.20
-------------------  -----------  ------------
Nodes                        381           416
Constraints                 1221          1188
Indexes                     2565          2518
Text indexes                 767           751
Matches                     3782          5286
Non-empty count             1994          3762
Non-empty rate               100           100
Bug reports                   22             9
Global state resets           16            19


## Bug Comparison with Neo4j Versions

In [14]:
bugs = ['ResultMismatchException', 'DatabaseCrashed']

In [15]:
bug_result_mismatch_exception_4 = [l for l in lines_4 if bugs[0] in l]
bug_result_mismatch_exception_5 = [l for l in lines if bugs[0] in l]
bug_database_crashed_4 = [l for l in lines_4 if bugs[1] in l]
bug_database_crashed_5 = [l for l in lines if bugs[1] in l]

In [16]:
print(tabulate([['ResultMismatchException', len(bug_result_mismatch_exception_4)/2, len(bug_result_mismatch_exception_5)/2],
                ['DatabaseCrashed', len(bug_database_crashed_4)/2, len(bug_database_crashed_5)/2]], 
               headers=['Type of bug', 'Neo4j 4.4', 'Neo4j 5.20']))

Type of bug                Neo4j 4.4    Neo4j 5.20
-----------------------  -----------  ------------
ResultMismatchException           12             8
DatabaseCrashed                   10             1
